# Baselines: risk methods, policies and hosting Qwen

This notebook compares the six risk methods and the decision thresholds on
labeled trajectories, offline. Collect trajectories first, with
`test_example.ipynb` section 5 or `sop-candidate collect --policy fixed` (see the quick start).
It also shows how to host the Qwen model yourself.

In [ ]:
from pathlib import Path
import json, os, sys

# Works whether Jupyter starts in the project folder or in scripts/.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "test_client").is_dir())
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data"
WORK = ROOT / ".runs" / "notebook"          # all outputs of this notebook (git-ignored)

# Your examiner and model connections. Set them as environment variables before
# starting Jupyter, or type them here (do not share notebooks containing tokens).
EXAMINER_URL = os.environ.get("SOP_EXAMINER_URL", "http://127.0.0.1:8801")
EXAMINER_TOKEN = os.environ.get("SOP_EXAMINER_TOKEN", "")
MODEL_URL = os.environ.get("SOP_MODEL_URL", "http://127.0.0.1:18080")
MODEL_TOKEN = os.environ.get("SOP_MODEL_TOKEN", "")

from baselines.runner import load_cases
from baselines.storage import read_jsonl
from baselines.uq import UQ_METHODS, Calibrator
from test_client.scoring import probability_metrics

labeled = load_cases(DATA)
index = {c["case_id"]: c for c in labeled}
RUN = WORK / "collect"   # a collect output with initial.jsonl
initial = read_jsonl(RUN / "initial.jsonl")
print(len(initial), "initial trajectories")

## Six risk methods

| method | idea | fitted on |
|---|---|---|
| `self_report` | the model's own error probability | – |
| `disagreement` | share of the 4 samples that differ from the answer, smoothed | – |
| `historical_failure` | error rate of the domain | `train` |
| `calibrated_self_report` | logistic calibration of the self-report | `cal_fit` |
| `calibrated_disagreement` | logistic calibration of disagreement | `cal_fit` |
| `supervised` | logistic model on trajectory features | `train` |

The comparison below uses `dev` cases, which no estimator was fitted on.

In [ ]:
calibrator = Calibrator().fit(labeled, initial)
dev = [s for s in initial if index[s["case_id"]]["training_split"] == "dev"]

def wrong(s):
    return int(s.get("failure") is not None or s.get("answer") != index[s["case_id"]]["label"])

errors = [wrong(s) for s in dev]
print(f"dev cases: {len(dev)}, initial error rate: {sum(errors) / max(1, len(errors)):.2f}")
for method in UQ_METHODS:
    risks = [1.0 if s.get("answer") is None else calibrator.predict(s, method) for s in dev]
    m = probability_metrics(risks, errors)
    print(f"{method:24s} brier={m.get('brier')} ap={m.get('ap')} ece={m.get('ece')}")

## Policies and DEFER thresholds

A policy decides which oracle facts to read and whether to answer (`AUTO`) or
hand over (`DEFER`, cost 1.0). An `AUTO` answer costs 0 when right, 1.5 for an
ordinary error and 5 for a severe one, so the best threshold is not 0.5.
`select_thresholds` picks the threshold with the lowest average loss on a
held-out split.

| policy | queries | decision |
|---|---|---|
| `no_oracle` | none | always AUTO |
| `all_defer` | none | always DEFER |
| `threshold` | none | DEFER when risk ≥ threshold |
| `fixed` | the first `fixed_count` legal fields | threshold |
| `all_facts` | every legal field within budget | threshold |
| `active` | only when risk ≥ threshold, most conflicting fields first | threshold |

In [ ]:
calibrator.select_thresholds(labeled, initial, split="cal_select")
for method, record in calibrator.selections.items():
    print(f"{method:24s} threshold={record['threshold']:.3f} auto share={record['coverage']:.2f} "
          f"loss={record['domain_macro_loss']:.3f}")

## A risk method of your own

Write a function from a trajectory to a probability and evaluate it the same
way. Trajectories contain the answer, the self-reports, the four samples, every
tool receipt (`history`) and the cost.

In [ ]:
def my_risk(snapshot):
    """Example: average of self-report and disagreement, more when tools conflicted."""
    if snapshot.get("answer") is None:
        return 1.0
    base = 0.5 * (calibrator.predict(snapshot, "calibrated_self_report")
                  + calibrator.predict(snapshot, "calibrated_disagreement"))
    receipts = [e for e in snapshot["history"] if e["kind"] == "tool_receipt"]
    return min(1.0, base + 0.02 * max(0, len(receipts) - 6))

m = probability_metrics([my_risk(s) for s in dev], errors)
print("my_risk", {k: m.get(k) for k in ("brier", "nll", "ece", "ap")})

## Hosting Qwen yourself (optional)

The examiner provides a shared model service. To run your own on a GPU, use a
separate Python 3.12 environment:

```bash
python -m venv .venv-model
.venv-model/bin/pip install -r src/qwen_server/requirements-model.lock.txt
.venv-model/bin/pip install -e ./src
mkdir -p .runs && cp src/qwen_server/runtime.M9.json .runs/runtime.M9.json
# then set model_path, and lower gpu_memory_utilization if the GPU is shared
export SOP_MODEL_TOKEN=$(python -c "import secrets; print(secrets.token_urlsafe(24))")
.venv-model/bin/sop-qwen serve --profile src/qwen_server/profiles/M9.json     --runtime .runs/runtime.M9.json
```

`sop-qwen download --profile src/qwen_server/profiles/M9.json --destination DIR`
fetches the exact weights; `serve` verifies them before starting.